# Day 02 — Data Analysis with Pandas

## أهداف اليوم

نتعرف على Pandas باعتبارها المكتبة الأساسية للتعامل مع البيانات الجدولية في هذا الجزء من المسار. سنتعلم إنشاء Series وDataFrame، قراءة البيانات وفحصها، اختيار الصفوف والأعمدة، الفلترة، التجميع، الدمج، وPivot Table.

> جرّب كل مثال بنفسك في Google Colab أو Jupyter. استخدم البيانات الصغيرة أولاً حتى تفهم شكل النتيجة قبل العمل على Dataset كبيرة.

## 1. ما هي Pandas؟

Pandas مكتبة Python للتعامل مع البيانات الجدولية، ويمكن تشبيه DataFrame بجدول Excel لكن يمكن التحكم به وتحليله باستخدام الكود.

لها بنيتان أساسيتان:

- **Series:** عمود واحد من القيم، وله فهرس (Index).
- **DataFrame:** جدول كامل يتكون من أعمدة متعددة، وكل عمود يمكن اعتباره Series.

سنستخدم الاختصار الشائع `pd` عند استيراد المكتبة.

In [ ]:
import pandas as pd

# Series: عمود واحد
s = pd.Series([10, 20, 30])
print(s)

# DataFrame: جدول من عدة أعمدة
df = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar"],
    "age": [22, 25, 30],
    "city": ["Gaza", "Gaza", "Rafah"]
})

display(df)

## 2. قراءة البيانات وأول نظرة عليها

عند استلام Dataset جديدة، ابدأ بفهم حجمها وأسماء أعمدتها وأنواع البيانات والقيم الناقصة.

| الأمر | ماذا يفعل؟ |
|---|---|
| `pd.read_csv("data.csv")` | قراءة ملف CSV |
| `df.head()` | عرض أول خمسة صفوف |
| `df.shape` | عدد الصفوف والأعمدة |
| `df.info()` | أنواع الأعمدة والقيم غير الناقصة |
| `df.describe()` | إحصاءات وصفية للأعمدة الرقمية |
| `df.columns` | أسماء الأعمدة |

هذه الأوامر الستة نقطة بداية عملية مع أي Dataset جديدة.

In [ ]:
# مثال: قراءة ملف CSV
# df = pd.read_csv("data.csv")

# نستخدم DataFrame الذي أنشأناه في القسم السابق للتجربة
print("First rows:")
display(df.head())

print("Shape:", df.shape)
print("\nInfo:")
df.info()

print("\nNumeric summary:")
display(df.describe())

print("Columns:", df.columns.tolist())

## 3. اختيار الأعمدة والصفوف

يوفر Pandas أكثر من طريقة للوصول إلى البيانات:

- `df["age"]`: اختيار عمود واحد ويعيد Series.
- `df[["name", "age"]]`: اختيار عدة أعمدة ويعيد DataFrame.
- `loc`: الاختيار باستخدام أسماء الفهارس أو الأعمدة.
- `iloc`: الاختيار باستخدام المواقع الرقمية.
- يمكن تمرير شرط منطقي إلى DataFrame لفلترة الصفوف.

عند الجمع بين أكثر من شرط استخدم `&` أو `|`، وضع كل شرط بين أقواس. لا تستخدم `and` و`or` مع Series.

In [ ]:
print(df["age"])
display(df[["name", "age"]])

print("loc:", df.loc[0, "name"])
print("iloc:", df.iloc[0, 1])

display(df[df["age"] > 23])

display(
    df[(df["age"] > 23) & (df["city"] == "Gaza")]
)

## 4. إضافة أعمدة وترتيب وعدّ القيم

يمكنك إنشاء عمود جديد اعتماداً على عمود موجود، وترتيب الصفوف، ومعرفة عدد مرات ظهور كل قيمة.

- `sort_values` لترتيب الصفوف حسب عمود.
- `value_counts` لحساب تكرار كل قيمة.
- `unique` لاستخراج القيم المختلفة دون تكرار.

In [ ]:
df["age_next_year"] = df["age"] + 1

display(df.sort_values("age", ascending=False))
print(df["city"].value_counts())
print(df["city"].unique())

## 5. التجميع — groupby

الفكرة الأساسية في `groupby` هي:

**Split → Apply → Combine**

1. تقسيم البيانات إلى مجموعات حسب عمود.
2. تطبيق عملية حسابية على كل مجموعة.
3. جمع النتائج في مخرجات منظمة.

مثلاً يمكن حساب متوسط العمر لكل مدينة، أو حساب عدد السجلات ومتوسط العمر لكل مجموعة.

In [ ]:
# متوسط العمر لكل مدينة
print(df.groupby("city")["age"].mean())

# أكثر من إحصائية في عملية واحدة
summary = df.groupby("city").agg(
    count=("name", "count"),
    avg_age=("age", "mean")
)

display(summary)

## 6. الدمج — merge

عملية `merge` تشبه JOIN في SQL. نربط جدولين باستخدام عمود مشترك، ونحدد نوع الدمج عبر `how`.

| how | النتيجة |
|---|---|
| `inner` | الصفوف التي لها تطابق في الجدولين |
| `left` | كل صفوف الجدول الأول مع البيانات المطابقة من الثاني |
| `right` | كل صفوف الجدول الثاني مع البيانات المطابقة من الأول |
| `outer` | كل الصفوف من الجدولين، سواء وُجد تطابق أم لا |

في المثال التالي نربط بيانات الأشخاص بجدول الطلبات باستخدام عمود `name`.

In [ ]:
orders = pd.DataFrame({
    "name": ["Ali", "Sara"],
    "total": [50, 80]
})

merged = pd.merge(df, orders, on="name", how="left")
display(merged)

## 7. الجدول المحوري — pivot_table

يساعدنا `pivot_table` على تلخيص البيانات في شكل جدول محوري. نحدد:

- `values`: العمود الذي نريد تلخيصه.
- `index`: العمود الذي سنقسم النتائج بناءً عليه.
- `aggfunc`: العملية الحسابية مثل `mean` أو `sum` أو `count`.

In [ ]:
pivot = df.pivot_table(
    values="age",
    index="city",
    aggfunc="mean"
)

display(pivot)

## 8. أخطاء شائعة

1. **SettingWithCopyWarning:** قد يظهر عند تعديل جزء مفلتر من DataFrame. استخدم `.copy()` عندما تريد نسخة مستقلة، أو استخدم `.loc` للتعديل الواضح.
2. **الدوال لا تعدّل الأصل دائماً:** كثير من الدوال تعيد DataFrame جديداً. احفظ النتيجة إذا أردت الاحتفاظ بها، مثل `df = df.dropna()`.
3. **أسماء الأعمدة:** انتبه للمسافات وحالة الأحرف، وتأكد من الاسم باستخدام `df.columns`.

In [ ]:
# مثال على نسخة مستقلة من البيانات المفلترة
adults = df[df["age"] >= 23].copy()
adults.loc[:, "age_next_year"] = adults["age"] + 1

display(adults)

## أسئلة مراجعة — Day 02

أجب دون النظر إلى الشرح:

1. ما الفرق بين Series وDataFrame؟
2. ما الفرق بين `loc` و`iloc`؟
3. كيف تختار عمودين فقط من DataFrame؟
4. كيف تحسب متوسط عمود لكل فئة في عمود آخر؟
5. متى تستخدم `how="left"` في `merge`؟
6. ما وظيفة `value_counts()`؟
7. لماذا قد نستخدم `.copy()` بعد الفلترة؟

**معيار الانتقال:** شغّل جميع الخلايا، ثم أجب عن الأسئلة السبعة. بعد ذلك تكون جاهزاً للتطبيق العملي على Titanic في اليوم الثالث.